# UNSB training on Kaggle (paisajes -> Monet)

Trains the `sb` mode of [Alherma7/UNSB](https://github.com/Alherma7/UNSB) for the *I'm Something of a Painter Myself* (`gan-getting-started`) competition, resumable across Kaggle sessions.

**Before running:**
1. Notebook Settings -> Accelerator -> GPU. Internet -> On (needed to `git clone`).
2. Add Data -> Competition Data -> `gan-getting-started`.
3. First run: leave `PREV_OUTPUT_DIR = None` in the config cell below.
4. At the end of a session, click **Save Version** ("Save & Run All"). This turns `/kaggle/working` into this notebook's own output dataset.
5. To continue training in a *new* edit session: Add Data -> "Notebook Output Files" -> this same notebook's latest saved version, then check `!ls /kaggle/input/` in the first code cell to find its mount name, and set `PREV_OUTPUT_DIR` to that path before re-running.

Each run trains for a wall-clock-bounded chunk (see `TIME_BUDGET_SECONDS`), so it's safe to just keep re-running/resuming across as many sessions as needed.

In [ ]:
!nvidia-smi
!ls /kaggle/input/

import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())

In [ ]:
# ============================== Configuration ==============================

# Set to the mount path of a previous "Save Version" output when resuming
# across sessions, e.g. '/kaggle/input/unsb-monet-training' (see step 5
# above -- check the `!ls /kaggle/input/` output from the cell above).
PREV_OUTPUT_DIR = None

RUN_NAME = 'monet_sb_v1'
TOTAL_N_EPOCHS = 60          # constant-LR phase length across the WHOLE multi-session arc
TOTAL_N_EPOCHS_DECAY = 60    # linear-decay phase length across the whole arc
VAL_SIZE = 24
SEED = 42

# Wall-clock budget for this session's training call. Kaggle GPU sessions
# are time-limited; this makes sure we always stop cleanly (with a full
# checkpoint saved) well before the platform kills the kernel, rather than
# risking a mid-epoch hard kill. Tune down if your session limit is shorter.
TIME_BUDGET_SECONDS = 6 * 3600

REPO_URL = 'https://github.com/Alherma7/UNSB.git'

In [ ]:
# ============================= Clone + deps =================================
import os

if not os.path.exists('/kaggle/working/UNSB'):
    !cd /kaggle/working && git clone {REPO_URL}
else:
    !cd /kaggle/working/UNSB && git pull

%cd /kaggle/working/UNSB
!pip install -q dominate pytorch-fid

In [ ]:
# ===================== Locate the competition data ==========================
# Kaggle has mounted the competition's monet_jpg/photo_jpg somewhere under
# /kaggle/input -- the exact mount path has varied historically, so search
# for it instead of hardcoding.
import glob

def find_dir(name):
    candidates = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert candidates, f'Could not find a "{name}" folder under /kaggle/input -- did you Add Data -> gan-getting-started?'
    return candidates[0]

MONET_DIR = find_dir('monet_jpg')
PHOTO_DIR = find_dir('photo_jpg')
print('MONET_DIR:', MONET_DIR, '(', len(os.listdir(MONET_DIR)), 'files)')
print('PHOTO_DIR:', PHOTO_DIR, '(', len(os.listdir(PHOTO_DIR)), 'files)')

In [ ]:
# ========================== Prepare the dataroot =============================
DATAROOT = '/kaggle/working/UNSB/datasets/monet2photo'

!python datasets/prepare_monet_dataset.py \
    --monet_dir "{MONET_DIR}" --photo_dir "{PHOTO_DIR}" \
    --out_dir "{DATAROOT}" --val_size {VAL_SIZE} --seed {SEED} \
    --expected_monet_count {len(os.listdir(MONET_DIR))} \
    --expected_photo_count {len(os.listdir(PHOTO_DIR))}

In [ ]:
# ======================= Resume from a previous session ======================
import re, shutil

CKPT_DIR = f'/kaggle/working/UNSB/checkpoints/{RUN_NAME}'
os.makedirs(CKPT_DIR, exist_ok=True)

resume_args = []
if PREV_OUTPUT_DIR is not None:
    prev_ckpt_dir = os.path.join(PREV_OUTPUT_DIR, 'UNSB', 'checkpoints', RUN_NAME)
    assert os.path.isdir(prev_ckpt_dir), f'No checkpoints found at {prev_ckpt_dir}'
    for f in os.listdir(prev_ckpt_dir):
        if f.endswith('.pth') or f == 'fid_log.csv' or f == 'loss_log.txt':
            shutil.copy2(os.path.join(prev_ckpt_dir, f), os.path.join(CKPT_DIR, f))

    completed_epochs = [int(m.group(1)) for f in os.listdir(CKPT_DIR)
                        if (m := re.match(r'(\d+)_net_G\.pth$', f))]
    assert completed_epochs, f'Copied files from {prev_ckpt_dir} but found no <epoch>_net_G.pth -- nothing to resume from'
    last_epoch = max(completed_epochs)
    print(f'Resuming after epoch {last_epoch}')
    resume_args = ['--continue_train', '--epoch_count', str(last_epoch + 1), '--epoch', str(last_epoch)]
else:
    print('Starting a fresh training run (PREV_OUTPUT_DIR is None)')

resume_args_str = ' '.join(resume_args)

In [ ]:
# ============================== Train (bounded) ==============================
# `timeout` sends SIGTERM to train.py once TIME_BUDGET_SECONDS elapses, so the
# session always ends cleanly. --save_epoch_freq 1 keeps a full <epoch>_net_*
# checkpoint after every completed epoch (the only thing the resume logic
# above trusts); anything mid-epoch when the timeout fires is simply redone
# next session, which is cheap relative to a full epoch.
!timeout {TIME_BUDGET_SECONDS} python train.py \
    --dataroot "{DATAROOT}" --name {RUN_NAME} --mode sb \
    --lambda_SB 1.0 --lambda_NCE 1.0 \
    --preprocess none --save_epoch_freq 1 \
    --n_epochs {TOTAL_N_EPOCHS} --n_epochs_decay {TOTAL_N_EPOCHS_DECAY} \
    --gpu_ids 0 --display_id -1 {resume_args_str}

In [ ]:
# ============================ Periodic FID eval ===============================
!python scripts/eval_checkpoints.py \
    --checkpoints_dir "{CKPT_DIR}" \
    --val_dir "{DATAROOT}/testA" \
    --monet_ref_dir "{MONET_DIR}" \
    --out_csv "{CKPT_DIR}/fid_log.csv" \
    --tmp_dir "{CKPT_DIR}/eval_tmp" \
    --gpu_ids 0

## Next steps

- Click **Save Version** now so this session's checkpoints/`fid_log.csv` become this notebook's output.
- If `fid_log.csv` is still improving and you haven't reached `TOTAL_N_EPOCHS + TOTAL_N_EPOCHS_DECAY`, open a new edit session, attach this notebook's own latest output as input data, set `PREV_OUTPUT_DIR` accordingly (see the instructions cell at the top), and re-run.
- Once you've picked a best epoch from `fid_log.csv` (plus a visual sanity check), use `scripts/kaggle_submission_notebook.py` as the basis for the actual graded submission notebook: point its `CHECKPOINT_PATH` at `<best_epoch>_net_G.pth` from this notebook's output, attach both this notebook's output and the `gan-getting-started` competition data, set `LIMIT = None`, and "Save & Run All (Commit)".